# Compare Kanana guardrails before and after LoRA

Serve the merged checkpoint from [notebook 4](4_finetune_guardrail_kanana_lora.ipynb) and replay the [notebook 3 benchmark](3_evaluate_guardrail_kanana.ipynb) against both NeMo configurations. Each row receives two decisions from the same Korean summarization request. A refusal in the final assistant message counts as a block, so the comparison includes both input and output rails.

Complete notebooks 3 and 4 in the same project. Run this notebook from a workbench with `oc` project access, the repository requirements, a ready baseline guardrail, and one available serving GPU alongside the baseline. Configure `GUARDRAILS_ROUTE` and the S3 settings in `.env`. For a private route CA, point `SSL_CERT_FILE` at its CA bundle.

If you submitted the YAML directly from the dashboard, download `training.json` from the plotting task's `analysis` artifact and set `KANANA_LORA_RUN` to its local directory.

## 1. Load the training run and the baseline configuration

Check the merged-model location and read the active NeMo configuration from notebook 3. Set `NUM_SAMPLES` to choose the prefix of its validation split.

In [ ]:
import base64
import hashlib
import json
import os
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

import httpx
import pandas as pd
import yaml
from IPython.display import display

lab_dir = Path.cwd() if (Path.cwd() / "kanana_lab.py").is_file() else Path.cwd() / "5_eval_guardrail"
sys.path.insert(0, str(lab_dir))
from kanana_lab import (
    EVAL_REVISION, PROMPT_TEMPLATE, ROOT, auth_token, guardrail_payload, is_refusal, load_lab_env, oc, required, s3_client,
)

load_lab_env()
NUM_SAMPLES = 200
if NUM_SAMPLES < 1:
    raise ValueError("Choose at least one evaluation row.")
namespace = required("NAMESPACE")
route = required("GUARDRAILS_ROUTE").rstrip("/")
run_dir = ROOT / required("KANANA_LORA_RUN")
summary = json.loads((run_dir / "training.json").read_text())
model_uri = urlparse(summary["model_uri"])
if model_uri.scheme != "s3" or model_uri.netloc != required("AWS_S3_BUCKET"):
    raise ValueError("Point the S3 settings at the bucket recorded by notebook 4.")
model_path = model_uri.path.strip("/")
s3_client().head_object(Bucket=model_uri.netloc, Key=f"{model_path}/config.json")

base_config = "guardrail-kanana-safety"
cr_name = "nemoguardrails"
baseline = json.loads(oc("get", "configmap", base_config, "-n", namespace, "-o", "json"))["data"]
baseline_yaml = yaml.safe_load(baseline["config.yaml"])
main_model = next(m["model"] for m in baseline_yaml["models"] if m["type"] == "main")
run_tag = hashlib.sha256(summary["model_uri"].encode()).hexdigest()[:10]
served_name = f"kanana-lora-{run_tag}"
tuned_config = f"guardrail-{served_name}"
out_dir = run_dir / f"comparison-{NUM_SAMPLES}"
out_dir.mkdir(exist_ok=True)
print(f"✅ Training run {run_dir.name} | {summary['data']['train_rows']:,} comments | {summary['base_model']}")
print(f"{base_config} → {tuned_config} | {summary['model_uri']}")

## 2. Serve the merged model on OpenShift AI

The KServe service loads the exact S3 path exported by notebook 4. Set `KANANA_S3_CONNECTION` to reuse a project data connection and `KANANA_SERVING_RUNTIME` to select an installed NVIDIA vLLM runtime. With the defaults, this cell creates a project connection and instantiates the OpenShift AI vLLM template if needed. [OpenShift AI S3 connections](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.5/html/working_on_projects/using-connections_projects)

In [ ]:
connection = os.getenv("KANANA_S3_CONNECTION") or "kanana-lora-storage"
runtime = os.getenv("KANANA_SERVING_RUNTIME") or "vllm-cuda-runtime"

def in_project(*args, stdin=None):
    return oc(*args, "-n", namespace, stdin=stdin)

# Read only the bucket field when checking an existing data connection.
bucket_field = in_project("get", "secret", connection, "--ignore-not-found", "-o", "jsonpath={.data.AWS_S3_BUCKET}")
if not bucket_field:
    if os.getenv("KANANA_S3_CONNECTION"):
        raise ValueError(f"Check the name and AWS_S3_BUCKET of connection {connection}.")
    secret = {
        "apiVersion": "v1", "kind": "Secret", "type": "Opaque",
        "metadata": {"name": connection, "annotations": {"opendatahub.io/connection-type-protocol": "s3"},
                     "labels": {"opendatahub.io/dashboard": "true"}},
        "stringData": {key: required(key) for key in
                       ("AWS_S3_ENDPOINT", "AWS_S3_BUCKET", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")},
    }
    secret["stringData"]["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION") or "us-east-1"
    in_project("create", "-f", "-", stdin=json.dumps(secret))
elif base64.b64decode(bucket_field).decode() != model_uri.netloc:
    raise ValueError(f"Connection {connection} points to a different S3 bucket.")

if not in_project("get", "servingruntime", runtime, "--ignore-not-found", "-o", "name"):
    if runtime != "vllm-cuda-runtime":
        raise ValueError(f"ServingRuntime {runtime} is missing from {namespace}.")
    template = oc("process", "vllm-cuda-runtime-template", "-n", "redhat-ods-applications", "-o", "json")
    in_project("apply", "-f", "-", stdin=template)

service = {
    "apiVersion": "serving.kserve.io/v1beta1", "kind": "InferenceService",
    "metadata": {
        "name": served_name, "labels": {"opendatahub.io/dashboard": "true", "networking.kserve.io/visibility": "cluster-local"},
        "annotations": {"serving.kserve.io/deploymentMode": "RawDeployment",
                        "opendatahub.io/connections": connection, "opendatahub.io/connection-path": model_path},
    },
    "spec": {"predictor": {"minReplicas": 1, "maxReplicas": 1, "model": {
        "runtime": runtime, "modelFormat": {"name": "vLLM"},
        "resources": {"requests": {"nvidia.com/gpu": "1", "cpu": "2", "memory": "20Gi"},
                      "limits": {"nvidia.com/gpu": "1", "cpu": "4", "memory": "32Gi"}},
    }}},
}
in_project("apply", "-f", "-", stdin=json.dumps(service))
in_project("wait", "--for=condition=Ready", f"inferenceservice/{served_name}", "--timeout=30m")
status = json.loads(in_project("get", "inferenceservice", served_name, "-o", "json"))["status"]
endpoint = (status.get("address") or {}).get("url") or status.get("url")
if not endpoint:
    # Raw deployments can expose only a predictor Service when ingress is disabled.
    predictor = json.loads(in_project("get", "service", f"{served_name}-predictor", "-o", "json"))
    ports = predictor["spec"]["ports"]
    port = next((p for p in ports if p.get("name", "").startswith("http") and "metrics" not in p["name"]),
                ports[0] if len(ports) == 1 else None)
    if not port:
        raise RuntimeError(f"Cannot identify the HTTP port of {served_name}-predictor.")
    scheme = "https" if port.get("name", "").startswith("https") else "http"
    endpoint = f"{scheme}://{served_name}-predictor.{namespace}.svc:{port['port']}"
address = urlparse(endpoint)
tuned_api = f"{address.scheme}://{address.netloc}/v1"
print(f"✅ {served_name} is ready at {tuned_api}")

## 3. Register a second NeMo configuration

Copy the deployed baseline's prompts, parser, input/output rails and main model. Point its safety model at the new cluster-local service and wait for the NeMo API to list both config IDs.

In [ ]:
tuned_files = dict(baseline)
config = yaml.safe_load(baseline["config.yaml"])
safety = next(m for m in config["models"] if m["type"] == "kanana_safety")
safety["model"] = served_name
safety.pop("api_key_env_var", None)
safety["parameters"].update(base_url=tuned_api, api_key="local-kserve")
tuned_files["config.yaml"] = yaml.safe_dump(config, sort_keys=False, allow_unicode=True)
in_project("apply", "-f", "-", stdin=json.dumps({
    "apiVersion": "v1", "kind": "ConfigMap", "metadata": {"name": tuned_config}, "data": tuned_files,
}))

resource = json.loads(in_project("get", "nemoguardrails", cr_name, "-o", "json"))
configs = resource["spec"]["nemoConfigs"]
if not any(c["name"] == base_config for c in configs):
    raise RuntimeError("Register the baseline by running notebook 3 first.")
entry = {"name": tuned_config, "configMaps": [tuned_config]}
if entry not in configs:
    if any(c["name"] == tuned_config for c in configs):
        raise RuntimeError(f"NeMo already uses {tuned_config} with a different ConfigMap.")
    patch = [{"op": "test", "path": "/metadata/resourceVersion", "value": resource["metadata"]["resourceVersion"]},
             {"op": "add", "path": "/spec/nemoConfigs/-", "value": entry}]
    in_project("patch", "nemoguardrails", cr_name, "--type=json", "-p", json.dumps(patch))

headers = {"Authorization": f"Bearer {auth_token()}"}
deadline = time.monotonic() + 300
with httpx.Client(headers=headers, timeout=15) as http:
    while True:
        try:
            response = http.get(f"{route}/v1/rails/configs")
            if response.status_code not in {502, 503, 504}:
                response.raise_for_status()
                available = {item["id"] for item in response.json()}
                if {base_config, tuned_config} <= available:
                    break
        except (httpx.NetworkError, httpx.TimeoutException):
            pass
        if time.monotonic() >= deadline:
            raise TimeoutError(f"NeMo did not mount both configs. Inspect deployment/{cr_name} in {namespace}.")
        time.sleep(5)
print("✅ Both NeMo configurations are available")

## 4. Select the paired evaluation rows

Take the first 200 rows by default, in the same order as notebook 3. Label 8 is negative and each hate category is positive. Each stored ID refers to the row index in the pinned validation split. [Evaluation dataset](https://huggingface.co/datasets/nayohan/K-MHaS)

In [ ]:
from datasets import load_dataset

valid = load_dataset("nayohan/K-MHaS", revision=EVAL_REVISION, split="valid")
sample = valid.select(range(min(NUM_SAMPLES, len(valid)))).to_pandas()
sample.index.name = "id"
sample = sample.reset_index()

def hate_label(value):
    labels = {int(label) for label in value.split(",")}
    if not labels <= set(range(9)) or (8 in labels and labels != {8}):
        raise ValueError(f"Unexpected validation labels: {value}")
    return int(labels != {8})

sample["actual"] = sample["label"].map(hate_label)
print(f"✅ {len(sample)} paired requests per model | {sample.actual.sum()} hate | {(sample.actual == 0).sum()} safe")

## 5. Replay both guardrails

Send the same chat completion to each config and apply the refusal phrases from notebook 3. Save each response and any request error so coverage can be checked before scoring.

In [ ]:
from tqdm.auto import tqdm

def classify(http, config_id, prompt):
    try:
        response = http.post(f"{route}/v1/chat/completions", json=guardrail_payload(main_model, config_id, prompt))
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
        if not isinstance(content, str) or not content.strip():
            raise ValueError("Missing completion content")
        filtered = is_refusal(content)
        return filtered, content[:200], None
    except (httpx.HTTPError, ValueError, TypeError, IndexError, KeyError) as exc:
        return None, None, str(exc)

rows = []
with httpx.Client(headers=headers, timeout=120) as http:
    for row in tqdm(sample.itertuples(index=False), total=len(sample), desc="Comparing guardrails"):
        pair = {"id": row.id, "document": row.document, "label": row.label, "actual": row.actual}
        prompt = PROMPT_TEMPLATE.format(document=row.document)
        for name, config_id in [("base", base_config), ("lora", tuned_config)]:
            pair[f"{name}_blocked"], pair[f"{name}_response"], pair[f"{name}_error"] = classify(http, config_id, prompt)
        rows.append(pair)

results = pd.DataFrame(rows)
results.to_csv(out_dir / "paired_predictions.csv", index=False)
paired = results.dropna(subset=["base_blocked", "lora_blocked"]).copy()
print(f"Paired coverage: {len(paired)}/{len(results)} | base errors: {results.base_error.notna().sum()}"
      f" | LoRA errors: {results.lora_error.notna().sum()}")
if paired.empty:
    raise RuntimeError(f"No valid pairs. Check the errors in {out_dir / 'paired_predictions.csv'}.")
print(f"✅ Saved predictions to {out_dir}")

## 6. Compare the confusion matrices

Both matrices use the same successfully scored rows and color scale. The difference is LoRA minus base in each cell. A negative FP change means fewer safe comments were blocked, and a negative FN change means fewer hateful comments passed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

truth = paired["actual"].astype(int)
matrices, metrics = {}, {}
for name in ("base", "lora"):
    pred = paired[f"{name}_blocked"].astype(int)
    cm = confusion_matrix(truth, pred, labels=[0, 1])
    matrices[name] = cm
    tn, fp, fn, tp = cm.ravel()
    metrics[name] = {
        "TN": tn, "FP": fp, "FN": fn, "TP": tp, "Accuracy": accuracy_score(truth, pred),
        "Precision": precision_score(truth, pred, zero_division=0), "Recall": recall_score(truth, pred, zero_division=0),
        "F1": f1_score(truth, pred, zero_division=0), "FPR": fp / (tn + fp) if tn + fp else np.nan,
        "FNR": fn / (tp + fn) if tp + fn else np.nan,
    }

scores = pd.DataFrame(metrics)
scores["LoRA − base"] = scores["lora"] - scores["base"]
scores.to_csv(out_dir / "metrics.csv")
display(scores.round(4))

delta = matrices["lora"] - matrices["base"]
scale = max(m.max() for m in matrices.values())
span = max(1, abs(delta).max())
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), layout="constrained")
for ax, title, values in zip(axes, ("Base Kanana", "LoRA Kanana", "LoRA − base"), (*matrices.values(), delta)):
    change = title == "LoRA − base"
    image = ax.imshow(values, cmap="RdBu_r" if change else "Blues", vmin=-span if change else 0,
                      vmax=span if change else scale)
    for (i, j), value in np.ndenumerate(values):
        r, g, b, _ = image.cmap(image.norm(value))
        color = "black" if 0.299*r + 0.587*g + 0.114*b > 0.56 else "white"
        text = f"{value:+d}" if change else str(value)
        tag = (("TN", "FP"), ("FN", "TP"))[i][j]
        ax.text(j, i, f"{tag}\n{text}", ha="center", va="center", color=color, fontsize=13)
    ax.set(xticks=[0, 1], xticklabels=["Allow", "Block"], yticks=[0, 1],
           yticklabels=["Not hate", "Hate"], xlabel="Guardrail decision", ylabel="Actual", title=title)
fig.suptitle(f"K-MHaS validation · {len(paired)}/{len(results)} paired comments")
fig.savefig(out_dir / "confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"✅ Saved confusion_matrices.png and metrics.csv in {out_dir}")

## 7. Inspect the decisions that changed

Review the corrected and newly introduced errors alongside the original K-MHaS labels. The prediction CSV includes all requested rows and both responses.

In [ ]:
changed = paired.loc[paired.base_blocked != paired.lora_blocked].copy()
fixed = changed.lora_blocked.astype(int) == changed.actual
changed["change"] = np.where(fixed, "Corrected by LoRA", "New error with LoRA")
changed.to_csv(out_dir / "changed_decisions.csv", index=False)
print(f"Corrected: {int(fixed.sum())} | New errors: {int((~fixed).sum())} | Unchanged: {len(paired) - len(changed)}")
display(changed[["id", "document", "label", "actual", "base_blocked", "lora_blocked", "change"]].head(10))
print(f"✅ Comparison saved in {out_dir}")
print(f"To release this serving GPU after the lab: oc delete inferenceservice {served_name} -n {namespace}")